# Example 12 — SCF with reconstructed-space vetoes

This notebook combines self-cross-feed migration with a veto applied after reconstruction. An SCF event migrated into a vetoed reconstructed bin is removed from the accepted PDF and from its normalization.


In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
from dalitzplotfitter import (DecayChannel, DecayModel, MassWindowVeto, NonResonant, RealImag, Resonance, SCFSignalPDF, SquareDalitzSCFMap, enable_x64)
from dalitzplotfitter.integration import GridIntegrator
enable_x64()
channel=DecayChannel('B+',('K+','pi+','pi-'))
model=DecayModel(channel,[Resonance('Kstar892',(0,2),RealImag(1,0),mass=0.8958,width=0.0474,spin=1),Resonance('rho770',(1,2),RealImag(0.65,0.10),mass=0.7753,width=0.1491,spin=1),NonResonant(RealImag(-0.5,0.1))],normalization_method='square-dalitz',normalization_resolution=280,normalization_pair=(0,2))


## 1. Build an SCF migration map


In [ ]:
NMP=NTP=22; n=NMP*NTP
mp=(np.arange(NMP)+0.5)/NMP; tp=(np.arange(NTP)+0.5)/NTP
gmp,gtp=np.meshgrid(mp,tp,indexing='ij'); coords=np.column_stack((gmp.ravel(),gtp.ravel()))
migration=np.empty((n,n))
for i,(m0,t0) in enumerate(coords):
    d2=((coords[:,0]-(m0+0.015))/0.045)**2+((coords[:,1]-t0)/0.055)**2
    row=np.exp(-0.5*d2); migration[i]=row/row.sum()
fraction=0.08+0.12*gmp.ravel()
scf_map=SquareDalitzSCFMap(jnp.asarray(migration),jnp.asarray(fraction),channel.parent_mass,channel.daughter_masses,NMP,NTP,pair=(0,2))


## 2. Apply a veto in reconstructed $K^+\pi^-$ mass


In [ ]:
veto=MassWindowVeto((0,2),1.80,1.95)
pdf_no_veto=SCFSignalPDF(lambda d,p:model.intensity(d,p),GridIntegrator(model.normalization_sample),scf_map)
pdf_veto=SCFSignalPDF(lambda d,p:model.intensity(d,p),GridIntegrator(model.normalization_sample),scf_map,veto=veto)
centres=scf_map.true_bin_data(); accepted=np.asarray(veto(centres))
print('normalization without veto:',float(pdf_no_veto.normalization({})))
print('normalization with veto:',float(pdf_veto.normalization({})))
print('accepted reconstructed-bin fraction:',accepted.mean())


## 3. Reconstructed density before and after the veto


In [ ]:
before=np.asarray(pdf_no_veto.numerator(centres,{})).reshape(NMP,NTP)
after=np.asarray(pdf_veto.numerator(centres,{})).reshape(NMP,NTP)
mask=accepted.reshape(NMP,NTP)
def draw(values,title):
    plt.figure(figsize=(6.2,5)); plt.imshow(values.T,origin='lower',aspect='auto',extent=(0,1,0,1)); plt.xlabel(r"$m'$"); plt.ylabel(r"$\theta'$"); plt.title(title); plt.colorbar(); plt.show()
draw(before,'SCF reconstructed density — no veto')
draw(mask.astype(float),'Reconstructed acceptance mask')
draw(after,'SCF reconstructed density — veto applied')


## 4. Dalitz-space view of the accepted reconstructed PDF


In [ ]:
sample=model.generate_phase_space(30000,seed=12001); vals=np.asarray(pdf_veto(sample.as_dict(),{})); keep=vals>0
plt.figure(figsize=(7,5.5)); h=plt.hist2d(np.asarray(sample.s13)[keep],np.asarray(sample.s23)[keep],bins=75,weights=vals[keep]); plt.colorbar(h[3],label='accepted reconstructed PDF weight'); plt.xlabel(r'$s_{13}$ [GeV$^2$]'); plt.ylabel(r'$s_{23}$ [GeV$^2$]'); plt.title('SCF + reconstructed-space veto'); plt.show()
